# Dataset

In [1]:
text = ""
with open("tinyshakesphere.txt") as f:
    text = f.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [2]:
vocab = list(set(text))
vocab_size = len(vocab)
print(vocab_size)

65


In [3]:
encode = {val:i for i,val in enumerate(vocab)}
decode = {i:val for i,val in enumerate(vocab)}

print(decode[0])

O


In [4]:
import torch
tokenized_text = torch.tensor([encode[c] for c in text],dtype=torch.long)
print(tokenized_text[:1000])

tensor([33, 64, 34, 36,  7,  5, 17, 64,  7, 64, 27, 32, 50, 21, 30, 25, 32, 43,
         4, 34, 32,  5, 40, 32,  5, 20, 34,  4, 60, 32, 32, 56,  5, 16, 50, 52,
         5, 43, 15, 34,  7, 35, 32, 34, 22,  5, 35, 32, 16, 34,  5, 41, 32,  5,
        36, 20, 32, 16, 53, 46, 30, 30, 26, 31, 31, 21, 30, 24, 20, 32, 16, 53,
        22,  5, 36, 20, 32, 16, 53, 46, 30, 30, 33, 64, 34, 36,  7,  5, 17, 64,
         7, 64, 27, 32, 50, 21, 30, 61,  4, 15,  5, 16, 34, 32,  5, 16, 31, 31,
         5, 34, 32, 36,  4, 31, 49, 32, 56,  5, 34, 16,  7, 35, 32, 34,  5,  7,
         4,  5, 56, 64, 32,  5,  7, 35, 16, 50,  5,  7,  4,  5, 43, 16, 41, 64,
        36, 35, 13, 30, 30, 26, 31, 31, 21, 30,  9, 32, 36,  4, 31, 49, 32, 56,
        46,  5, 34, 32, 36,  4, 31, 49, 32, 56, 46, 30, 30, 33, 64, 34, 36,  7,
         5, 17, 64,  7, 64, 27, 32, 50, 21, 30, 33, 64, 34, 36,  7, 22,  5, 52,
         4, 15,  5, 53, 50,  4, 40,  5, 17, 16, 64, 15, 36,  5, 37, 16, 34, 60,
        64, 15, 36,  5, 64, 36,  5, 60, 

In [5]:
train_data = tokenized_text[:int(0.9*len(text))]
val_data = tokenized_text[int(0.9*len(text)):]

print(len(train_data),len(val_data))

1003854 111540


In [6]:
block_size = 128

def get_batch(split):
    data = train_data if split == 'train' else val_data

    random_starting_indices = torch.randint(len(data) - block_size, (block_size,))

    x = torch.stack([data[i:i+block_size] for i in random_starting_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_starting_indices])
    return x,y
    

x,y = get_batch("train")
print(x)
print(y)


tensor([[22,  5,  7,  ...,  5, 50,  4],
        [50, 52,  5,  ..., 50, 56,  5],
        [43, 32, 50,  ...,  4, 49,  4],
        ...,
        [31, 36, 30,  ..., 35, 32, 34],
        [30, 30, 37,  ..., 56,  5, 35],
        [35, 32,  5,  ..., 64, 32, 31]])
tensor([[ 5,  7,  4,  ..., 50,  4, 34],
        [52,  5, 52,  ..., 56,  5, 43],
        [32, 50, 56,  ..., 49,  4, 36],
        ...,
        [36, 30, 14,  ..., 32, 34, 46],
        [30, 37, 26,  ...,  5, 35,  4],
        [32,  5,  3,  ..., 32, 31, 56]])


In [21]:
print(torch.tril(torch.ones(128, 128)))

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 1., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])


# Transformer Model

In [28]:
import torch.nn as nn
import torch.nn.functional as F

class AttentionHead(nn.Module):
    def __init__(self, d_model, head_size, max_seq_length):
        super().__init__()
        self.input_dim = d_model

        self.query = nn.Linear(d_model, head_size, bias=False)
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        
        self.head_size = head_size
        
        # Mask buffer
        self.register_buffer('tril', torch.tril(torch.ones(max_seq_length, max_seq_length))) #(max_seq_length, max_seq_length)
        
    def forward(self, x):
            B,T,C = x.shape

            # Q @ K_T is "read only", thus we can apply mask after this step
            
            Q = self.query(x) # (B, T, head_size)
            K = self.key(x)   # (B, T, head_size)
            V = self.value(x) # (B, T, head_size)
            
            K_T = K.transpose(-2, -1) # (B, head_size, T)
            
            scores = Q @ K_T # (B,T,head_size) @ (B,head_size,T) -> (B, T, T)
            scores = scores / (self.head_size ** 0.5) 
 
            scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            
            weights = F.softmax(scores, dim=-1) # (B, T, T)     
            out = weights @ V # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
            
            return out

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_length):
        super().__init__()
        self.head_size = d_model // n_heads

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.heads = nn.ModuleList(
            [AttentionHead(d_model, self.head_size, max_seq_length) for _ in range(n_heads)]
        )
        self.proj = nn.Linear(d_model, d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        # Pre-LN
        x_norm = self.ln1(x)
        
        head_outputs = [head(x_norm) for head in self.heads]
        mha_out = torch.cat(head_outputs, dim=-1)
        proj_out = self.proj(mha_out)
        
        # Pure residual addition (No normalization here)
        x = x + proj_out

        x_norm2 = self.ln2(x)
        ffn_out = self.ffn(x_norm2)
        x = x + ffn_out

        return x
        

class LanguageModel(nn.Module):
    def __init__(self,vocab_size, d_model, max_seq_length, n_heads, n_layers): 
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.pos_embedding = nn.Embedding(num_embeddings=max_seq_length,embedding_dim=d_model)

        self.layers = nn.Sequential(
            *[TransformerBlock(d_model, n_heads, max_seq_length) for _ in range(n_layers)]
        )

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.max_seq_length = max_seq_length


    def forward(self,input_encodings,targets_encodings=None):
        B,T = input_encodings.shape

        positions = torch.arange(T, device=input_encodings.device)

        input_embeddings = self.embedding(input_encodings) + self.pos_embedding(positions)
        transform = self.layers(input_embeddings)
        out = self.lm_head(transform) # (B,T,vocab_size)

        loss = None
        if targets_encodings is not None:
            out_for_loss = out.permute(0,2,1) # (B,vocab_size,T)
            loss = F.cross_entropy(out_for_loss,targets_encodings)
        
        return loss, out 
    
    def generate(self,input_encoding, max_output_len):
        for _ in range(max_output_len):
            input_cropped = input_encoding[:, -self.max_seq_length:]
            _,logits = self(input_cropped)
            logits = logits[:,-1,:] # Last char of sequence
            predictions_probs = F.softmax(logits,dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding,prediction),dim=1)

        return input_encoding
    
model = LanguageModel(vocab_size, d_model=512, max_seq_length=128, n_heads=8, n_layers=8) 

start_context = torch.zeros((1, 1), dtype=torch.long)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))



O!!rhM&QCM:3-'blPWpLi:fcLg-.IGKM.!;:gVUHRUTGKY:lLsb,ZQVeJLCZdrooMSwbCluvuCoxQMO wttQ3M?BxEyERmbGgz,!v


# Training Transformer

In [29]:
device = "cuda" if torch.cuda.is_available else 'cpu'
print(device)

model = model.to(device)

cuda


In [30]:
def train(model, n_steps, optimizer, device):
    model.train()
    
    for step in range(n_steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()       
        loss, _ = model(x, y)   
        loss.backward()         
        optimizer.step()            
        
        if step % 5 == 0:
            print(f"Step {step} | Train Loss = {loss.item():.4f}")


import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
train(model, 400, optimizer,device)

Step 0 | Train Loss = 4.6557
Step 5 | Train Loss = 3.3431
Step 10 | Train Loss = 2.9506
Step 15 | Train Loss = 2.8704
Step 20 | Train Loss = 2.7423
Step 25 | Train Loss = 2.7126
Step 30 | Train Loss = 2.6443
Step 35 | Train Loss = 2.6202
Step 40 | Train Loss = 2.5889
Step 45 | Train Loss = 2.5689
Step 50 | Train Loss = 2.5333
Step 55 | Train Loss = 2.5162
Step 60 | Train Loss = 2.5148
Step 65 | Train Loss = 2.5042
Step 70 | Train Loss = 2.5051
Step 75 | Train Loss = 2.4996
Step 80 | Train Loss = 2.4799
Step 85 | Train Loss = 2.4952
Step 90 | Train Loss = 2.4898
Step 95 | Train Loss = 2.4807
Step 100 | Train Loss = 2.4674
Step 105 | Train Loss = 2.4543
Step 110 | Train Loss = 2.4680
Step 115 | Train Loss = 2.4467
Step 120 | Train Loss = 2.4480
Step 125 | Train Loss = 2.4485
Step 130 | Train Loss = 2.4436
Step 135 | Train Loss = 2.4563
Step 140 | Train Loss = 2.4254
Step 145 | Train Loss = 2.4363
Step 150 | Train Loss = 2.4280
Step 155 | Train Loss = 2.4149
Step 160 | Train Loss = 2.4095

In [ ]:
def evaluate(model,device):
    model.eval()
    x,y = get_batch('eval')
    x, y = x.to(device), y.to(device)

    loss,_ = model(x,y)
    print("Validation Loss = ",loss.item())

evaluate(model,device)
start_context = torch.zeros((1, 1), dtype=torch.long).to(device)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))


Validation Loss =  2.1323399543762207
O, lom Lement with: the cone ofeertsell
He: is derpleay ant ofel quecan mowabamban'd th: nor,
A hat h


In [36]:
inp_text = "He:"
inp_text_tokenized = torch.tensor([encode[c] for c in inp_text], dtype=torch.long).unsqueeze(0).to(device)
gen = model.generate(inp_text_tokenized, 100)
print(''.join([decode[int(i)] for i in gen[0]]))

He:
Why my stay for hat nor magh sly nowht ler,
Jerod read, to do and swans beast I a for med bearrend



In [32]:

torch.save(model.state_dict(), 'model_weights.pt')
print("Model successfully saved!")

Model successfully saved!
